# GVQCC 28.09 frontier — C=43 / 42 GPU optimizer

Uses the unchanged public `maoxin123` 15-loader chemistry circuit, FeMoCo official verification, the user's C43/C42 code from `corcondor/gvqcc` PR #9. **This notebook does not modify the verifier or automatically submit a result**. No C43/C42 pass is claimed before the actual run. Attribution for the public KMB remains maoxin123 / Boristown / upstream credited authors (Apache-2.0 / CC0 as applicable). The existing DQI562 and 44Q invsqrt are a separate, already-verified foundation.

Attach the **official Google Quantum AI compilation competition input** and the public 15-loader family (`maoxin123/qcc-frontier-dqi-chem-mirror`) under **Add Input**. To use GPU choose Accelerator in notebook settings; some GPUs have slow float64 performance.


In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
import numpy as np
root=Path('/kaggle/working/gvqcc_repo');root.mkdir(parents=True,exist_ok=True)
verify_files=list(Path('/kaggle/input').rglob('verifiers/dfthcsos/verify.py'))
if not verify_files: raise RuntimeError('Attach official competition verifiers input (verifiers/dfthcsos/verify.py).')
verifiers_root=verify_files[0].parents[1]
kickmix_roots=[p.parent for p in Path('/kaggle/input').rglob('kickmix.so')]
for extra in [root,verifiers_root,*kickmix_roots]: sys.path.insert(0,str(extra))
os.environ['PYTHONPATH']=os.pathsep.join([str(root),str(verifiers_root),*[str(x) for x in kickmix_roots],os.environ.get('PYTHONPATH','')])
print('Official verifiers:',verifiers_root,'kickmix paths:',kickmix_roots)


In [ ]:
SOURCES = {"inverse_v2_chemistry_screen.py":"\"\"\"Reference-free feasibility bounds and C43 initializations from a passing fit.\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / 'official_sources/Inzainos/verifiers'))\nfrom dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors, reconstruct_x, lambda_h2\n\n\ndef h2_change_norm(x, y):\n    \"\"\"Accumulate directly in blocks to avoid cancellation in Gram traces.\"\"\"\n    x, y = x.reshape(-1, x.shape[-2] * x.shape[-1]), y.reshape(-1, y.shape[-2] * y.shape[-1])\n    total = 0.0\n    for lo in range(0, x.shape[1], 384):\n        hi = min(x.shape[1], lo + 384)\n        d = y[:, lo:hi].T @ y - x[:, lo:hi].T @ x\n        total += np.einsum('ij,ij->', d, d)\n    return float(np.sqrt(total))\n\n\ndef h2_coefs(data, brot):\n    r, b, c = data['W'].shape\n    u = reconstruct_u_vectors(data['angle_words'] / (1 << (brot - 1)), data['tree_edges'])\n    return reconstruct_x(u[76:].reshape(r, b, 76), data['W'])\n\n\ndef c43_householder(data, x):\n    out = dict(data)\n    weights, identity, eigens, dropped, transforms = [], [], [], [], []\n    for r in range(len(x)):\n        xx = x[r].reshape(x.shape[1], -1)\n        vals, vecs = np.linalg.eigh(xx @ xx.T)\n        v = vecs[:, 0]\n        j = int(np.argmax(np.abs(v)))\n        e = np.zeros_like(v)\n        e[j] = -np.copysign(1.0, v[j])\n        u = v - e\n        u /= np.linalg.norm(u)\n        q = np.eye(len(v)) - 2 * np.outer(u, u)\n        keep = [k for k in range(len(v)) if k != j]\n        q = q[:, keep]\n        weights.append(data['W'][r] @ q)\n        identity.append(data['H2_I'][r] @ q)\n        eigens.append(vals[:8].tolist())\n        dropped.append({'r':r,'closest_column':j,'maximum_v_component':float(np.max(np.abs(v))),'norm_squared':float(vals[0])})\n        transforms.append(q)\n    out['W'], out['H2_I'] = np.array(weights), np.array(identity)\n    return out, {'smallest_eigenvalues':eigens,'dropped_components':dropped}\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument('--outdir',type=Path,required=True)\n    args=p.parse_args()\n    args.outdir.mkdir(parents=True,exist_ok=True)\n    source=ROOT/'dqi_inputs/chemistry_4033/submission.npz'\n    data=dict(np.load(source));data.update(dict(np.load(source.with_name('asterios_refit.npz'))))\n    base=h2_coefs(data,8)\n    rows=[]\n    for brot in (7,6):\n        divisor=1<<(8-brot)\n        for mode in ('floor','half_up','nearest_even'):\n            dd=dict(data)\n            if mode=='floor':words=data['angle_words']//divisor\n            elif mode=='half_up':words=(data['angle_words']+divisor//2)//divisor\n            else:words=np.rint(data['angle_words']/divisor).astype(np.int64)\n            dd['angle_words']=words%(1<<brot)\n            change=h2_change_norm(base,h2_coefs(dd,brot))\n            row={'kind':f'brot{brot}_{mode}','h2_change_norm':change,'error2_lower_bound_from_passing_baseline':max(0,change-.020),'definitely_fails_at_fixed_W':change>.040}\n            rows.append(row)\n            print(json.dumps(row),flush=True)\n    dd,detail=c43_householder(data,base)\n    xx=h2_coefs(dd,8)\n    change=h2_change_norm(base,xx)\n    offset=data['H2_I']-data['W'].sum(1)\n    newoffset=dd['H2_I']-dd['W'].sum(1)\n    row={'kind':'C43_smallest_H2_component_householder','h2_change_norm':change,'error2_lower_bound_from_passing_baseline':max(0,change-.020),'definitely_fails_before_refit':change>.040,'lambda_h2_before':lambda_h2(data['W'],offset),'lambda_h2_after':lambda_h2(dd['W'],newoffset),**detail}\n    rows.append(row);print(json.dumps(row),flush=True)\n    np.savez_compressed(args.outdir/'c43_householder.npz',**dd)\n    (args.outdir/'screen.json').write_text(json.dumps(rows,indent=2)+'\\n')\n\n\nif __name__=='__main__':main()\n","inverse_v2_chemistry_refit.py":"\"\"\"Bounded fixed-angle C43 refit with analytic derivatives and exact reference.\n\nThis is an experiment, not a submission generator. Final candidates are\nrechecked by the unchanged official numerical evaluator. H2 residuals are\nexpanded about the passing baseline to avoid cancellation of large norms.\n\"\"\"\nfrom __future__ import annotations\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment, minimize\n\nfrom inverse_v2_chemistry_screen import ROOT\nfrom dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors, symmetric_from_lower, evaluate_dfthc_angle_words\nfrom dfthcsos.scoreinfra.reference import check_fcidump, load_reference, load_fixed_shifts, UPPER_ENERGY\n\n\nclass FixedAngles:\n    def __init__(self, baseline):\n        t=time.monotonic()\n        self.base=baseline\n        self.R,self.B,self.C0=baseline['W'].shape\n        self.N=baseline['U'].shape[-1]\n        shifts=load_fixed_shifts()\n        self.S=np.asarray(symmetric_from_lower(shifts['Hsym_lower'],self.N),dtype=float)\n        self.beta=float(shifts['H1_I'])-.5*np.trace(self.S)\n        self.ref=load_reference(check_fcidump())\n        self.hbase=self.ref['h1_thc']-.5*self.ref['electrons']*self.S+self.beta*np.eye(self.N)\n        self.gap_const=UPPER_ENERGY-self.ref['ecore']+self.ref['electrons']*self.beta\n        u=reconstruct_u_vectors(baseline['angle_words']/128,baseline['tree_edges'])\n        u/=np.linalg.norm(u,axis=1)[:,None]\n        self.one=u[:self.N]\n        self.outer=np.einsum('ip,iq->ipq',u[self.N:],u[self.N:])\n        rows,cols=np.tril_indices(self.N)\n        self.rows,self.cols=rows,cols\n        self.scale=np.where(rows==cols,1.0,np.sqrt(2.0))\n        self.V=self.outer[:,rows,cols]*self.scale\n        self.G=self.V@self.V.T\n        self.G4=self.G.reshape(self.R,self.B,self.R,self.B).transpose(0,2,1,3)\n        self.P0=baseline['W']@baseline['W'].transpose(0,2,1)\n        x=np.einsum('rbc,rbp->rcp',baseline['W'],self.V.reshape(self.R,self.B,-1),optimize=True).reshape(-1,len(rows))\n        target=self.ref['tei'][rows[:,None],cols[:,None],rows[None,:],cols[None,:]].copy()\n        sl=self.S[rows,cols]\n        ident=(rows==cols).astype(float)\n        target+=.5*(np.outer(sl,ident)+np.outer(ident,sl))\n        target*=self.scale[:,None]*self.scale[None,:]\n        residual=x.T@x-target\n        self.e20=float(np.einsum('ij,ij->',residual,residual))\n        projected=self.V@residual\n        self.A=np.array([projected[r*self.B:(r+1)*self.B]@self.V[r*self.B:(r+1)*self.B].T for r in range(self.R)])\n        self.calls=0\n        print(json.dumps({'prepared_seconds':time.monotonic()-t,'baseline_error2':self.e20**.5}),flush=True)\n\n    def evaluate(self, packed, c, *, smooth=1e-7):\n        self.calls+=1\n        size=self.R*self.B*c\n        w=packed[:size].reshape(self.R,self.B,c)\n        ii=packed[size:].reshape(self.R,c)\n        d=ii-w.sum(1)\n        x=np.einsum('rbc,rbpq->rcpq',w,self.outer.reshape(self.R,self.B,self.N,self.N),optimize=True)\n        hp=self.hbase-np.einsum('rc,rcpq->pq',d,x)\n        vals,vec=np.linalg.eigh(hp)\n        rr,cc=linear_sum_assignment(-np.abs(self.one@vec))\n        match=np.empty(self.N,dtype=int);match[rr]=cc\n        qrec=np.einsum('j,jp,jq->pq',vals[match],self.one,self.one,optimize=True)\n        e1=qrec-hp\n        err1sq=float(np.einsum('ij,ij->',e1,e1))\n        evweights=np.empty(self.N)\n        evweights[match]=np.einsum('jp,pq,jq->j',self.one,e1,self.one,optimize=True)\n        ae1=2*((vec*evweights)@vec.T-e1)\n        sign=(vec*np.sign(vals))@vec.T\n        lh1=float(np.abs(vals).sum())\n        absw=np.sqrt(w*w+smooth*smooth) if smooth else np.abs(w)\n        absi=np.sqrt(ii*ii+smooth*smooth) if smooth else np.abs(ii)\n        l1=absw.sum(1)+absi\n        lh2=float(.25*np.sum(l1*l1))\n        lam=lh1+lh2\n        gap=float(self.gap_const+lh1-vals.sum()+.5*np.sum(d*d))\n        eff=float(np.sqrt(max(1e-100,gap*(2*lam-gap))))\n\n        def hgrad(a):\n            ax=np.einsum('pq,rcpq->rc',a,x)\n            av=np.einsum('pq,rbpq->rb',a,self.outer.reshape(self.R,self.B,self.N,self.N))\n            return ax[:,None,:]-av[:,:,None]*d[:,None,:],-ax\n\n        gw1,gi1=hgrad(ae1)\n        gwl,gil=hgrad(sign)\n        sw=w/absw if smooth else np.sign(w)\n        si=ii/absi if smooth else np.sign(ii)\n        gwl+=.5*l1[:,None,:]*sw\n        gil+=.5*l1*si\n        gwg,gig=hgrad(sign-np.eye(self.N))\n        gwg-=d[:,None,:]\n        gig+=d\n        gwe=gap/eff*gwl+(lam-gap)/eff*gwg\n        gie=gap/eff*gil+(lam-gap)/eff*gig\n\n        delta=w@w.transpose(0,2,1)-self.P0\n        kd=np.zeros_like(self.A)\n        for r in range(self.R):\n            for s in range(self.R):\n                g=self.G4[r,s]\n                kd[r]+=g@delta[s]@g.T\n        err2sq=float(self.e20+2*np.einsum('rij,rij->',self.A,delta)+np.einsum('rij,rij->',kd,delta))\n        gw2=4*(self.A+kd)@w\n        def pack(a,b):return np.r_[a.ravel(),b.ravel()]\n        constraints=np.array([err1sq/(.01999998**2)-1,err2sq/(.01999998**2)-1,(3.91489802-gap)/.01])\n        derivs=np.array([pack(gw1,gi1)/(.01999998**2),pack(gw2,np.zeros_like(ii))/(.01999998**2),-pack(gwg,gig)/.01])\n        return eff,pack(gwe,gie),constraints,derivs,{'lambda_eff_smooth':eff,'Lambda_smooth':lam,'gap':gap,'error1':err1sq**.5,'error2':max(err2sq,0)**.5}\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument('candidate',type=Path)\n    p.add_argument('--seconds',type=float,default=150)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--iterations',type=int,default=80)\n    args=p.parse_args()\n    base=dict(np.load(ROOT/'dqi_inputs/chemistry_4033/submission_asterios_full.npz'))\n    data=dict(np.load(args.candidate))\n    model=FixedAngles(base)\n    c=data['W'].shape[2]\n    x=np.r_[data['W'].ravel(),data['H2_I'].ravel()]\n    result=model.evaluate(x,c)\n    print(json.dumps({'initial':result[-1],'constraints':result[2].tolist()}),flush=True)\n    # Check analytic gradients in nonzero directions before trusting the fit.\n    rng=np.random.default_rng(20261114)\n    d=rng.normal(size=x.shape);d/=np.linalg.norm(d)\n    step=1e-6\n    plus=model.evaluate(x+step*d,c);minus=model.evaluate(x-step*d,c)\n    fd=np.r_[(plus[0]-minus[0])/(2*step),(plus[2]-minus[2])/(2*step)]\n    an=np.r_[result[1]@d,result[3]@d]\n    rel=float(np.max(np.abs(fd-an)/np.maximum(1,np.abs(fd))))\n    print(json.dumps({'gradient_directional_fd':fd.tolist(),'analytic':an.tolist(),'scaled_max_error':rel}),flush=True)\n    if rel>1e-3:raise RuntimeError('analytic gradient check failed')\n    start=time.monotonic();lag=np.zeros(3);trace=[];latest=x.copy()\n    class Budget(Exception):pass\n    for outer,penalty in enumerate((2.,10.,50.,250.,1250.)):\n        def objective(xx):\n            if time.monotonic()-start>args.seconds:raise Budget()\n            eff,g,con,cg,metrics=model.evaluate(xx,c)\n            shifted=np.maximum(0,lag+penalty*con)\n            f=eff+(np.sum(shifted**2)-np.sum(lag**2))/(2*penalty)\n            grad=g+shifted@cg\n            return f,grad\n        def accepted_step(xx):\n            nonlocal latest\n            latest=xx.copy()\n        try:\n            latest=x.copy()\n            opt=minimize(objective,x,jac=True,method='L-BFGS-B',callback=accepted_step,options={'maxiter':args.iterations,'maxcor':10,'ftol':1e-13,'gtol':1e-8,'maxls':25})\n            x=opt.x\n            msg=str(opt.message)\n        except Budget:\n            x=latest;msg='time budget reached'\n        eff,g,con,cg,metrics=model.evaluate(x,c)\n        lag=np.maximum(0,lag+penalty*con)\n        row={'outer':outer,'penalty':penalty,'metrics':metrics,'constraints':con.tolist(),'seconds':time.monotonic()-start,'calls':model.calls,'termination':msg}\n        trace.append(row);print(json.dumps(row),flush=True)\n        if time.monotonic()-start>args.seconds:break\n    size=model.R*model.B*c\n    data['W']=x[:size].reshape(model.R,model.B,c)\n    data['H2_I']=x[size:].reshape(model.R,c)\n    metrics=evaluate_dfthc_angle_words(data['angle_words'],data['W'],data['H2_I']-data['W'].sum(1),model.S,model.beta,model.ref,brot=8,tree_edges=data['tree_edges'],energy_upper=UPPER_ENERGY)\n    args.out.parent.mkdir(parents=True,exist_ok=True)\n    np.savez_compressed(args.out,**data)\n    report={'source':str(args.candidate),'official_numeric_metrics':metrics,'trace':trace,'gradient_check_scaled_error':rel,'full_circuit_verified':False}\n    args.out.with_suffix('.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps({'official_numeric_metrics':metrics}),flush=True)\n\n\nif __name__=='__main__':main()\n","rank43_42_joint_gpu.py":"#!/usr/bin/env python3\n\"\"\"C=43/42 official 8-bit angle words, CPU/CUDA numerical search.\n\nThe source NPZ is an already verified C44 family. This script changes only W\nand H2_I, not the 15 ANGLE loaders or the quantized word table.\nIt does not submit to Kaggle. All finalists are rechecked by the unchanged\nofficial numerical evaluator; optional full ANGLE verification is explicit.\n\"\"\"\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for b in iter(lambda: f.read(1 << 20), b\"\"):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef resource_cost(rank, r, b, n, j, loader_t, loader_temp):\n    from dfthcsos.scoreinfra.costs_prepare import SpinFreeData_V1\n    p = SpinFreeData_V1(R=r, B=b, C=rank, N=n, chi=10, beta=8)\n    t = p.outer_state_prep() + p.inner_state_prep() + p.givens_rotations() + 2 * loader_t\n    qo, qi = p.outer_state_prep_qubit_cost(), p.inner_state_prep_qubit_cost()\n    q = qo[\"persistent\"] + qi[\"persistent\"] + 8 * j + max(\n        qo[\"temporary\"], qi[\"temporary\"], loader_temp\n    )\n    return int(t), int(q)\n\n\ndef compressed_start(base, gram, rank, seed, jitter=0.0):\n    \"\"\"Projector-Gram low-rank initializer with native d=H2_I-sum(W).\"\"\"\n    w0, i0 = base[\"W\"], base[\"H2_I\"]\n    r, b, c0 = w0.shape\n    assert 1 <= rank <= c0 and i0.shape == (r, c0)\n    rng = np.random.default_rng(seed)\n    w = np.empty((r, b, rank))\n    d = np.empty((r, rank))\n    native = i0 - w0.sum(axis=1)\n    for g in range(r):\n        wc = w0[g].copy()\n        if jitter:\n            wc += jitter * np.linalg.norm(wc) / np.sqrt(wc.size) * rng.normal(size=wc.shape)\n        eig, vec = np.linalg.eigh(wc.T @ gram[g, g] @ wc)\n        u = vec[:, np.argsort(eig)[-rank:]]\n        w[g], d[g] = wc @ u, native[g] @ u\n    return w, d\n\n\ndef as_candidate(base, w, d):\n    return {**base, \"W\": w, \"H2_I\": d + w.sum(axis=1)}\n\n\ndef passes(m):\n    return (\n        m[\"error_1\"] <= .020 and m[\"error_2\"] <= .020 and\n        m[\"Lambda\"] > 49 and m[\"E_gap\"] > 3.914898 and\n        0 < m[\"E_gap\"] < 2 * m[\"Lambda\"] and\n        math.isfinite(m[\"lambda_eff\"])\n    )\n\n\nclass GpuModel:\n    \"\"\"Float64 surrogate using *actual* fixed-angle word projector tensors.\"\"\"\n    def __init__(self, model, device):\n        import torch\n        self.torch, self.device = torch, torch.device(device)\n        def t(x):\n            return torch.as_tensor(np.array(x, copy=True), device=device, dtype=torch.float64)\n        self.r, self.b, _ = model.base[\"W\"].shape\n        self.n = model.N\n        self.g4, self.a, self.p0 = t(model.G4), t(model.A), t(model.P0)\n        self.e20 = t(model.e20)\n        self.outer = t(model.outer.reshape(self.r, self.b, self.n, self.n))\n        self.hbase, self.one, self.gap_const = t(model.hbase), t(model.one), t(model.gap_const)\n\n    def metrics(self, w, d):\n        torch = self.torch\n        ii = d + w.sum(dim=1)\n        delta = w @ w.transpose(1, 2) - self.p0\n        kd = torch.zeros_like(delta)\n        for s in range(self.r):\n            gs = self.g4[:, s]\n            kd = kd + gs @ delta[s] @ gs.transpose(1, 2)\n        e2sq = self.e20 + 2 * (self.a * delta).sum() + (kd * delta).sum()\n        x = torch.einsum(\"rbc,rbpq->rcpq\", w, self.outer)\n        h = self.hbase - torch.einsum(\"rc,rcpq->pq\", d, x)\n        eig, vec = torch.linalg.eigh(h)\n        from scipy.optimize import linear_sum_assignment\n        rows, cols = linear_sum_assignment(-np.abs((self.one @ vec).detach().cpu().numpy()))\n        match = np.empty(self.n, dtype=int)\n        match[rows] = cols\n        qrec = torch.einsum(\n            \"j,jp,jq->pq\",\n            eig[torch.as_tensor(match, device=self.device)], self.one, self.one\n        )\n        error1 = torch.linalg.matrix_norm(qrec - h)\n        error2 = torch.sqrt(torch.clamp(e2sq, min=1e-30))\n        l1 = torch.abs(w).sum(dim=1) + torch.abs(ii)\n        lam1 = torch.abs(eig).sum()\n        lam = lam1 + .25 * (l1 * l1).sum()\n        gap = self.gap_const + lam1 - eig.sum() + .5 * (d * d).sum()\n        eff = torch.sqrt(torch.clamp(gap * (2 * lam - gap), min=1e-30))\n        return {\"error_1\": error1, \"error_2\": error2,\n                \"Lambda\": lam, \"E_gap\": gap, \"lambda_eff\": eff}\n\n    def objective(self, w, d, penalty):\n        torch = self.torch\n        m = self.metrics(w, d)\n        violation = sum(\n            torch.relu(m[k] / .0199 - 1)**2 for k in (\"error_1\", \"error_2\")\n        )\n        violation += torch.relu((3.914899 - m[\"E_gap\"]) / .01)**2\n        violation += torch.relu((49.00001 - m[\"Lambda\"]) / 10)**2\n        return torch.log(m[\"lambda_eff\"]) + penalty * violation\n\n\ndef run(args):\n    import torch\n    from inverse_v2_chemistry_refit import FixedAngles\n    from dfthcsos.scoreinfra.dfthc_math import evaluate_dfthc_angle_words\n    from dfthcsos.scoreinfra.reference import UPPER_ENERGY\n\n    base_path = Path(args.base_npz)\n    base = dict(np.load(base_path, allow_pickle=False))\n    fixed = FixedAngles(base)\n    r, b, c = base[\"W\"].shape\n    if (r, b, c) != (8, 143, 44):\n        raise ValueError(f\"Expected 8x143x44, got {(r, b, c)}\")\n    t44, q44 = resource_cost(44, r, b, fixed.N, args.j, args.loader_t, args.loader_temp)\n    if t44 != args.assert_t44 or q44 != args.assert_q44:\n        raise ValueError(f\"Public loader mismatch: T={t44}, Q={q44}\")\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=False)\n    torch.set_num_threads(1)\n    model = GpuModel(fixed, args.device)\n    rows = []\n\n    for rank in args.ranks:\n        t, q = resource_cost(rank, r, b, fixed.N, args.j, args.loader_t, args.loader_temp)\n        for trial in range(args.seeds):\n            w0, d0 = compressed_start(\n                base, fixed.G4, rank, 20261116 + 100 * rank + trial,\n                jitter=0.0 if trial == 0 else min(.015, .003 * trial)\n            )\n            if args.warmstart_dir:\n                checkpoint = Path(args.warmstart_dir) / f\"C{rank}_seed{trial}.npz\"\n                if checkpoint.exists():\n                    init = dict(np.load(checkpoint, allow_pickle=False))\n                    if init[\"W\"].shape != (r, b, rank) or init[\"H2_I\"].shape != (r, rank):\n                        raise ValueError(f\"Warmstart shape mismatch: {checkpoint}\")\n                    w0 = init[\"W\"]\n                    d0 = init[\"H2_I\"] - w0.sum(axis=1)\n            w = torch.nn.Parameter(torch.as_tensor(w0.copy(), device=args.device, dtype=torch.float64))\n            d = torch.nn.Parameter(torch.as_tensor(d0.copy(), device=args.device, dtype=torch.float64))\n            optimizer = torch.optim.Adam([w, d], lr=args.lr)\n            best = None\n            def record(step):\n                nonlocal best\n                with torch.no_grad():\n                    m = {k: float(v) for k, v in model.metrics(w, d).items()}\n                excess = sum(max(0.0, m[k] / .020 - 1) for k in (\"error_1\", \"error_2\"))\n                excess += max(0., (3.914898 - m[\"E_gap\"]) / .01)\n                excess += max(0., (49.0 - m[\"Lambda\"]) / 10)\n                key = (excess > 0, excess, m[\"lambda_eff\"] * t * q)\n                if best is None or key < best[0]:\n                    best = (key, w.detach().cpu().numpy().copy(),\n                            d.detach().cpu().numpy().copy(), step)\n            record(0)\n            for step in range(1, args.steps + 1):\n                penalty = [1., 10., 100., 1000., 10000.][min(4, (step - 1) * 5 // args.steps)]\n                optimizer.zero_grad(set_to_none=True)\n                loss = model.objective(w, d, penalty)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError(f\"Nonfinite objective C={rank}, seed={trial}, step={step}\")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_([w, d], 10.)\n                optimizer.step()\n                if step % args.check_each == 0 or step == args.steps:\n                    record(step)\n            _, wbest, dbest, best_step = best\n            cand = as_candidate(base, wbest, dbest)\n            exact = evaluate_dfthc_angle_words(\n                cand[\"angle_words\"], cand[\"W\"], cand[\"H2_I\"] - cand[\"W\"].sum(1),\n                fixed.S, fixed.beta, fixed.ref, brot=8,\n                tree_edges=cand[\"tree_edges\"], energy_upper=UPPER_ENERGY\n            )\n            chemistry_cost = float(exact[\"lambda_eff\"] * t * q)\n            eligible = passes(exact) and chemistry_cost < args.max_chemistry_cost\n            stem = f\"C{rank}_seed{trial}\"\n            path = out / f\"{stem}.npz\"\n            np.savez_compressed(path, **cand)\n            report = {\n                \"rank\": rank, \"trial\": trial, \"step\": best_step,\n                \"T_model\": t, \"Q_model\": q, \"cost_model\": chemistry_cost,\n                \"official_numeric_metrics\": exact,\n                \"source_sha256\": digest(base_path), \"candidate_sha256\": digest(path),\n                \"numeric_eligible\": eligible, \"full_native_verified\": False,\n                \"status\": \"NUMERIC_ONLY\" if eligible else \"REJECT\"\n            }\n            if eligible and args.angle_dir:\n                from dfthcsos.verify import verify\n                full = verify(Path(args.angle_dir), path, tree_edges=cand[\"tree_edges\"])\n                report[\"official_native_receipt\"] = full\n                ok = (full.get(\"status\") == \"PASS\" and\n                      int(full.get(\"verified_shots\", 0)) >= 78080 and\n                      int(full.get(\"toffoli_count\", -1)) == t and\n                      int(full.get(\"num_qubits\", -1)) == q and\n                      float(full.get(\"score\", float(\"inf\"))) < args.max_chemistry_cost)\n                report[\"full_native_verified\"] = ok\n                report[\"status\"] = \"OFFICIAL_PASS\" if ok else \"REJECT_RESOURCE_OR_COST\"\n            (out / f\"{stem}.json\").write_text(\n                json.dumps(report, indent=2, allow_nan=False) + \"\\n\"\n            )\n            rows.append({k: report[k] for k in\n                (\"rank\", \"trial\", \"cost_model\", \"numeric_eligible\", \"full_native_verified\", \"status\")})\n            print(json.dumps(rows[-1]), flush=True)\n    (out / \"summary.json\").write_text(\n        json.dumps({\"rows\": rows, \"source\": digest(base_path),\n                    \"device\": args.device, \"verified_submission\": False}, indent=2) + \"\\n\"\n    )\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--base-npz\", required=True)\n    p.add_argument(\"--output-dir\", required=True)\n    p.add_argument(\"--angle-dir\")\n    p.add_argument(\"--warmstart-dir\")\n    p.add_argument(\"--device\", choices=(\"cpu\", \"cuda\"), default=\"cuda\")\n    p.add_argument(\"--ranks\", type=int, nargs=\"+\", choices=(43, 42), default=[43, 42])\n    p.add_argument(\"--steps\", type=int, default=1200)\n    p.add_argument(\"--seeds\", type=int, default=3)\n    p.add_argument(\"--check-each\", type=int, default=25)\n    p.add_argument(\"--lr\", type=float, default=2e-4)\n    p.add_argument(\"--loader-t\", type=int, default=4025)\n    p.add_argument(\"--loader-temp\", type=int, default=77)\n    p.add_argument(\"--j\", type=int, default=5)\n    p.add_argument(\"--assert-t44\", type=int, default=14712)\n    p.add_argument(\"--assert-q44\", type=int, default=427)\n    p.add_argument(\"--max-chemistry-cost\", type=float, default=294762732.98)\n    args = p.parse_args()\n    if min(args.steps, args.seeds, args.check_each, args.loader_t) <= 0:\n        p.error(\"positive count parameters required\")\n    if not (math.isfinite(args.lr) and args.lr > 0):\n        p.error(\"positive finite learning rate required\")\n    import torch\n    if args.device == \"cuda\" and not torch.cuda.is_available():\n        p.error(\"CUDA not available; choose --device cpu\")\n    run(args)\n\n\nif __name__ == \"__main__\":\n    main()\n","frontier_score_gate.py":"#!/usr/bin/env python3\n\"\"\"QCC official component-cost arithmetic; conditional, not verified score.\n\nRounding in leaderboard badges is separate from the underlying official\nverifier receipts. No C43 or C42 row is a valid circuit until official PASS.\n\"\"\"\nfrom __future__ import annotations\nimport argparse\nimport json\nimport math\n\nREF = {\n    \"invsqrt\": 2076774.233715,\n    \"dqi\": 24439463583672.93,\n    \"dfthcsos\": 3355599588.3155375,\n}\nPUBLIC_28_09 = {\n    \"invsqrt\": 75006.38,\n    \"dqi\": 347375470571.73,\n    \"dfthcsos\": 294762732.98,\n}\nOUR_VERIFIED_LOCAL = {\n    \"invsqrt\": 75170.13631745959,\n    \"dqi\": 347373921975.41064,\n    \"dfthcsos\": 294763061.58129317,\n}\nCHEM_T = {44: 14712, 43: 14634, 42: 14555}\nCHEM_Q = 427\n\n\ndef gm(costs):\n    if set(costs) != set(REF) or not all(\n        math.isfinite(x) and x > 0 for x in costs.values()\n    ):\n        raise ValueError(\"Need all three positive finite component costs\")\n    return math.exp(sum(math.log(REF[k] / costs[k]) for k in REF) / 3)\n\n\ndef c_scenarios(costs=None, target=28.3):\n    base = dict(PUBLIC_28_09 if costs is None else costs)\n    if not (math.isfinite(target) and target > 0):\n        raise ValueError(\"Bad target score\")\n    old = gm(base)\n    m = base[\"dfthcsos\"] / (CHEM_T[44] * CHEM_Q)\n    required_factor = (old / target)**3\n    chemistry_target = base[\"dfthcsos\"] * required_factor\n    rows = []\n    for c, t in CHEM_T.items():\n        conditional = {**base, \"dfthcsos\": m * t * CHEM_Q}\n        rows.append({\n            \"C\": c,\n            \"T_same_loaders\": t,\n            \"Q_same_workspace\": CHEM_Q,\n            \"lambda_held_fixed\": m,\n            \"score_if_numeric_and_native_pass\": gm(conditional),\n            \"lambda_needed_for_target\": chemistry_target / (t * CHEM_Q),\n            \"lambda_drop_needed_percent\": 100 * (1 - chemistry_target / (t * CHEM_Q) / m),\n            \"status\": \"COST_ONLY_NOT_VERIFIED\",\n        })\n    return {\"baseline_score\": old, \"target\": target,\n            \"chemistry_cost_needed_if_other_components_held\": chemistry_target,\n            \"rows\": rows}\n\n\ndef dqi_qubit_tradeoff(q_old, t_old, r_ratio=1.0, q_new=880):\n    \"\"\"Maximum Toffoli with new width, holding R*Q^2*T <= old cost.\"\"\"\n    if min(q_old, q_new) <= 0 or not all(\n        math.isfinite(x) and x > 0 for x in (t_old, r_ratio)\n    ):\n        raise ValueError(\"Bad Q/T/R values\")\n    tmax = t_old * (q_old / q_new)**2 / r_ratio\n    return {\n        \"Q_old\": q_old, \"Q_new\": q_new, \"T_old\": t_old,\n        \"R_new_over_old\": r_ratio,\n        \"T_new_max_to_not_worsen_DQI\": tmax,\n        \"extra_T_budget\": tmax - t_old,\n    }\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--target\", type=float, default=28.3)\n    args = p.parse_args()\n    hybrid = {**PUBLIC_28_09, \"dqi\": OUR_VERIFIED_LOCAL[\"dqi\"]}\n    print(json.dumps({\n        \"public_reported_score_model\": gm(PUBLIC_28_09),\n        \"our_local_official_score\": gm(OUR_VERIFIED_LOCAL),\n        \"hybrid_source_cost_score_if_reverified\": gm(hybrid),\n        \"hybrid_verification\": \"NOT_YET_REBUILT_OR_REVERIFIED\",\n        \"C43_C42_cost_only\": c_scenarios(hybrid, args.target),\n        \"DQI_Q_T_tradeoffs_if_R_FIXED\": [\n            dqi_qubit_tradeoff(890, 368137.84857421875, q_new=q)\n            for q in (889, 885, 880, 875, 870)\n        ],\n    }, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n"}
for name,src in SOURCES.items():
    p=root/name; p.write_text(src,encoding='utf-8')
    print(name,len(src),'bytes SHA256',hashlib.sha256(src.encode()).hexdigest()[:12])


In [ ]:
import torch
from dfthcsos.scoreinfra.costs_prepare import SpinFreeData_V1
print('PyTorch:',torch.__version__,'CUDA available:',torch.cuda.is_available())
family=[]
for p in Path('/kaggle/input').rglob('submission.npz'):
    if len(list(p.parent.glob('angle_k_*.kmb')))!=15: continue
    try:
        with np.load(p,allow_pickle=False) as d:
            if d['W'].shape!=(8,143,44):continue
    except Exception:continue
    family.append(p)
pin='f6e12f023a20f7aa44b7e981bc399d535c3920a2bc579bcea0aa713a6337367d'
matched=[p for p in family if hashlib.sha256(p.read_bytes()).hexdigest()==pin]
if len(matched)==1: base=matched[0]
elif len(family)==1: base=family[0]
else: raise RuntimeError('Select a unique C44 public family. Found '+str([str(x) for x in family]))
print('Pinned C44 factorization:',base)


In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'
command=[sys.executable,str(root/'rank43_42_joint_gpu.py'),
  '--base-npz',str(base),'--angle-dir',str(base.parent),
  '--output-dir','/kaggle/working/gvqcc_c43_c42_v1',
  '--ranks','43','42','--device',device,'--steps','1200','--seeds','3',
  '--loader-t','4025','--assert-t44','14712','--assert-q44','427',
  '--max-chemistry-cost','294762732.98']
print('RUNNING:', ' '.join(command))
with subprocess.Popen(command,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
    for line in p.stdout: print(line,end='')
    if p.wait(): raise RuntimeError('C43/C42 run exited with failure')


In [ ]:
out=Path('/kaggle/working/gvqcc_c43_c42_v1')
report=json.loads((out/'summary.json').read_text())
for row in report['rows']:
    print(row)
print('OFFICIAL_PASS count:',sum(r['status']=='OFFICIAL_PASS' for r in report['rows']))
print('No Kaggle competition submission was made by this notebook.')